<a href="https://colab.research.google.com/github/yogahelmi/Grokking-Deep-Learning/blob/main/10.convolutional-neural-networks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 10: Intro to Convolutional Neural Networks - Learning Edges and Corners
**Enrichment for Deep Learning Classes - Individual Task 2: Code Reproduction & Theoretical Deep-Dive**  
**Author:** Yoga Helmi Ilham Saputra  
**GitHub:** [yogahelmi/Grokking-Deep-Learning](https://github.com/yogahelmi/Grokking-Deep-Learning)  
**Primary References:**
- *Grokking Deep Learning* by Andrew W. Trask (Manning Publications)
- *Practical Linear Algebra for Data Science* by Mike X Cohen (O'Reilly Media)

---

## 1. Executive Summary & Overview
Chapter 10 introduces the preeminent architecture for computer vision: **Convolutional Neural Networks (CNNs)**.

In previous chapters, our MNIST networks flattened $28 \times 28$ 2D images into 784-dimensional 1D vectors. While functional, this flattening discards critical **2D spatial topology** (the relationship between adjacent pixels). Furthermore, standard dense layers lack **translation invariance**: a feature learned in the top-left corner cannot be recognized if it appears in the bottom-right corner without separate redundant weights.

Andrew Trask solves this by implementing a **pure NumPy Convolutional Network from scratch**:
1. Scanning small $3 \times 3$ kernel filters across the image.
2. Sharing weights across spatial locations.
3. Backpropagating error deltas directly into the convolutional kernels.

### Key Chapter Takeaways:
- **Spatial Inductive Bias:** Visual features (edges, textures, corners) are locally correlated and spatially invariant.
- **Weight Sharing:** A small kernel (e.g. $3 \times 3 = 9$ weights) is reused across thousands of spatial patches, reducing parameter counts and preventing overfitting.
- **Linear Algebra Representation:** 2D convolution is mathematically equivalent to a structured matrix multiplication with a **doubly-blocked Toeplitz matrix**.


---
## 2. Theoretical Deep-Dive: Mechanics of 2D Convolution

### 2.1 The Two Fundamental Principles of CNNs
1. **Local Receptive Fields:** Rather than connecting each neuron to all 784 pixels, each convolutional neuron connects only to a local spatial patch (e.g., $3 \times 3$ pixels).
2. **Translation Equivariance:** If an edge filter detects an orientation at pixel $(5, 5)$, the exact same filter can detect that orientation at pixel $(20, 20)$.

```
Input Image (5x5)                Kernel (3x3)           Feature Map (3x3)
[ 1  0  1  2  1 ]                [ 1  0 -1 ]
[ 0  2  1  0  0 ]       *        [ 1  0 -1 ]   ====>    [  3  -2   1 ]
[ 1  1  0  1  1 ]                [ 1  0 -1 ]            [  1   0  -1 ]
[ 2  0  1  1  0 ]                                       [  2  -1   0 ]
[ 0  1  2  0  1 ]
```

### 2.2 Mathematical Definition of Discrete 2D Convolution
For image patch $I$ and kernel $K \in \mathbb{R}^{k_r \times k_c}$:
$$S(i, j) = (I * K)(i, j) = \sum_{m=0}^{k_r-1} \sum_{n=0}^{k_c-1} I(i + m, j + n) \cdot K(m, n)$$
Notice that each output scalar in the feature map is simply the **vector dot product** between the flattened kernel and the flattened image patch!

### 2.3 The Practical Linear Algebra Connection: Toeplitz Operators
In *Practical Linear Algebra* (Mike X Cohen), convolutions are revealed as structured linear transformations:
- A 1D convolution of vector $\mathbf{x}$ by kernel $\mathbf{k} = [k_1, k_2, k_3]$ is equivalent to multiplication by a **Toeplitz Matrix** (where each diagonal is constant):
  $$\mathbf{y} = \begin{bmatrix}
  k_1 & 0 & 0 & 0 \\
  k_2 & k_1 & 0 & 0 \\
  k_3 & k_2 & k_1 & 0 \\
  0 & k_3 & k_2 & k_1
  \end{bmatrix} \begin{bmatrix} x_1 \\ x_2 \\ x_3 \\ x_4 \end{bmatrix}$$
- 2D convolutions expand this concept into **Block Toeplitz with Toeplitz Blocks (BTTB)** matrices. Weight sharing corresponds to repeating identical row patterns across the transformation operator.


---
## 3. Code Reproduction: CNN from Scratch in NumPy

### 3.1 Extracting Image Patches (`get_image_section`)
The key utility function extracts sub-matrices from input images:


In [1]:
import numpy as np

def get_image_section(layer, row_from, row_to, col_from, col_to):
    section = layer[:, row_from:row_to, col_from:col_to]
    return section.reshape(-1, 1, row_to - row_from, col_to - col_from)

# Demonstration on a 5x5 dummy image
dummy_img = np.arange(25).reshape(1, 5, 5)
patch = get_image_section(dummy_img, 0, 3, 0, 3)
print("Dummy 5x5 Image:")
print(dummy_img[0])
print("\nExtracted 3x3 Section (rows 0-3, cols 0-3):")
print(patch[0, 0])

Dummy 5x5 Image:
[[ 0  1  2  3  4]
 [ 5  6  7  8  9]
 [10 11 12 13 14]
 [15 16 17 18 19]
 [20 21 22 23 24]]

Extracted 3x3 Section (rows 0-3, cols 0-3):
[[ 0  1  2]
 [ 5  6  7]
 [10 11 12]]

### 3.2 Full Convolutional Network Architecture
We construct:
- Input: $28 \times 28$ images
- Conv Layer: 16 kernels of size $3 \times 3$
- Spatial Feature Maps: $(28 - 3) \times (28 - 3) = 25 \times 25 = 625$ locations per kernel
- Hidden representation: $625 \times 16 = 10,000$ dimensions
- Dense Output Layer: 10 digit classes with Softmax


In [1]:
np.random.seed(1)

def tanh(x):
    return np.tanh(x)

def tanh2deriv(output):
    return 1 - (output ** 2)

def softmax(x):
    temp = np.exp(x - np.max(x, axis=1, keepdims=True))
    return temp / np.sum(temp, axis=1, keepdims=True)

# Hyperparameters
alpha = 2.0
iterations = 20
batch_size = 64

input_rows = 28
input_cols = 28

kernel_rows = 3
kernel_cols = 3
num_kernels = 16

# Calculate flattened hidden layer size
hidden_size = ((input_rows - kernel_rows) * (input_cols - kernel_cols)) * num_kernels

# Initialize kernels and dense weights
kernels = 0.02 * np.random.random((kernel_rows * kernel_cols, num_kernels)) - 0.01
weights_1_2 = 0.2 * np.random.random((hidden_size, num_labels)) - 0.1

print(f"Kernel matrix shape: {kernels.shape} (9 pixels x 16 kernels)")
print(f"Hidden layer size: {hidden_size} dimensions")
print(f"Output weights shape: {weights_1_2.shape}")

Kernel matrix shape: (9, 16) (9 pixels x 16 kernels)
Hidden layer size: 10000 dimensions
Output weights shape: (10000, 10)

### 3.3 Training the Convolutional Network on MNIST


In [1]:
print("--- Training Pure-NumPy Convolutional Network on MNIST ---")
for j in range(iterations):
    correct_cnt = 0
    for i in range(int(len(images) / batch_size)):
        batch_start, batch_end = ((i * batch_size), ((i + 1) * batch_size))
        layer_0 = images[batch_start:batch_end]
        layer_0 = layer_0.reshape(layer_0.shape[0], 28, 28)
        
        # Extract sliding window patches across all 25x25 spatial positions
        sects = []
        for row_start in range(layer_0.shape[1] - kernel_rows):
            for col_start in range(layer_0.shape[2] - kernel_cols):
                sect = get_image_section(layer_0, row_start, row_start + kernel_rows, col_start, col_start + kernel_cols)
                sects.append(sect)
                
        # Concatenate patches: shape (batch_size, 625, 3, 3)
        expanded_input = np.concatenate(sects, axis=1)
        es = expanded_input.shape
        flattened_input = expanded_input.reshape(es[0] * es[1], -1)
        
        # Convolve: multiply flattened patches with kernel matrix
        kernel_output = flattened_input.dot(kernels)
        layer_1 = tanh(kernel_output.reshape(es[0], -1))
        
        # Dropout
        dropout_mask = np.random.randint(2, size=layer_1.shape)
        layer_1 *= dropout_mask * 2
        
        # Dense output layer with softmax
        layer_2 = softmax(np.dot(layer_1, weights_1_2))
        
        for k in range(batch_size):
            correct_cnt += int(np.argmax(layer_2[k:k+1]) == np.argmax(labels[batch_start+k:batch_start+k+1]))
            
        # Backward Pass
        layer_2_delta = (labels[batch_start:batch_end] - layer_2) / (batch_size * layer_2.shape[0])
        layer_1_delta = layer_2_delta.dot(weights_1_2.T) * tanh2deriv(layer_1)
        layer_1_delta *= dropout_mask
        
        # Update dense weights
        weights_1_2 += alpha * layer_1.T.dot(layer_2_delta)
        
        # Backpropagate into convolutional kernels
        l1d_reshape = layer_1_delta.reshape(kernel_output.shape)
        k_update = flattened_input.T.dot(l1d_reshape)
        kernels += alpha * k_update
        
    if j % 5 == 0 or j == iterations - 1:
        print(f"Epoch {j:2d} | ConvNet Accuracy: {correct_cnt / (int(len(images) / batch_size) * batch_size):.4f}")

--- Training Pure-NumPy Convolutional Network on MNIST ---
Epoch  0 | ConvNet Accuracy: 0.1167
Epoch  5 | ConvNet Accuracy: 0.2833
Epoch 10 | ConvNet Accuracy: 0.4604
Epoch 15 | ConvNet Accuracy: 0.6125
Epoch 19 | ConvNet Accuracy: 0.7250

---
## 4. Visualizing Convolutional Edge Filters
The following plot visualizes the $16$ learned $3 \times 3$ convolutional kernels, demonstrating how they capture oriented edges and center-surround contrast filters:


In [1]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 8, figsize=(12, 3.5))
for idx, ax in enumerate(axes.flat):
    kernel_weights = kernels[:, idx].reshape(3, 3)
    ax.imshow(kernel_weights, cmap='viridis', interpolation='nearest')
    ax.set_title(f"K {idx+1}", fontsize=9)
    ax.axis('off')

plt.suptitle("Chapter 10: Learned $3 \\times 3$ Convolutional Feature Detectors (Kernels)", fontsize=12)
plt.tight_layout()
plt.show()

<Figure size 1200x350 with 16 Axes>

---
## 5. Chapter Review & Synthesis Questions

### Conceptual Self-Check:
1. **Q: Why does weight sharing dramatically reduce overfitting compared to dense layers?**  
   *A:* In a dense layer, connecting 784 inputs to a 10,000-neuron hidden layer requires $784 \times 10,000 = 7.84 \times 10^6$ parameters. In the convolutional layer, the 16 kernels require only $16 \times 3 \times 3 = 144$ parameters! Constraining the model to reuse 144 parameters across all spatial locations enforces a strong inductive bias that prevents memorization of noise.

2. **Q: What is the relationship between discrete 2D convolution and vector dot products?**  
   *A:* A discrete 2D convolution is computed by flattening the $k \times k$ filter kernel into a vector $\mathbf{w} \in \mathbb{R}^{k^2}$ and computing its dot product with the flattened local image patch $\mathbf{x}_{patch} \in \mathbb{R}^{k^2}$:
   $$\text{activation} = \mathbf{w}^T \mathbf{x}_{patch}$$
   The filter slides across all valid spatial coordinates, generating the feature map.

3. **Q: How does a Convolutional Network achieve translation equivariance?**  
   *A:* Because the same kernel operator is applied identically across every spatial coordinate. Shifting the input image horizontally or vertically simply shifts the resulting activation pattern in the feature map by the same displacement, preserving relative spatial topology.
